In [1]:
import pandas as pd

In [3]:
bitcoin = pd.read_csv('data/bitcoin-hourly-ohclv-dataset/btc_hourly_ohclv_ta.csv')

In [4]:
bitcoin['dt'] = pd.to_datetime(bitcoin['DATETIME'])
bitcoin.head(1)

,UNIX_TIMESTAMP,DATETIME,OPEN,HIGH,CLOSE,LOW,VOLUME,SMA_20,EMA_12,EMA_26,...,VOLUME_SMA,MFI,ATR,PRICE_CHANGE,HIGH_LOW_RATIO,CLOSE_OPEN_RATIO,VOLATILITY_30D,PRICE_VOLATILITY_30D,HL_VOLATILITY_30D,dt
0,1418623200,2014-12-15 06:00:00.000,349.78,350.07,349.54,348.81,287499.35,348.784,348.845388,348.741338,...,202156.2935,41.596185,1.917314,-0.000686,1.003612,0.999314,0.036548,13.498893,0.009256,2014-12-15 06:00:00


In [5]:
!pip install plotly


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# def make_data(df):
    
thing = bitcoin.loc[bitcoin.DATETIME.str.startswith('2015-12-15')]

In [7]:
import plotly.graph_objects as go


fig = go.Figure(data=[go.Candlestick(x=thing['DATETIME'],
                open=thing['OPEN'],
                high=thing['HIGH'],
                low=thing['LOW'],
                close=thing['CLOSE'])])

In [8]:
fig.show()

In [9]:
bitcoin['date'] = bitcoin['dt'].dt.date

In [10]:
dfs_by_date = {d: sub_df for d, sub_df in bitcoin.groupby("date")}


In [11]:
df_dict = {}
count = 0 
for i in dfs_by_date:

    df_new = bitcoin.loc[bitcoin.date == i,:]
    df_dict[count] = df_new
    count +=1

In [12]:
df_dict

{0:     UNIX_TIMESTAMP                 DATETIME    OPEN    HIGH   CLOSE     LOW  \
 0       1418623200  2014-12-15 06:00:00.000  349.78  350.07  349.54  348.81   
 1       1418626800  2014-12-15 07:00:00.000  349.54  351.10  345.20  344.74   
 2       1418630400  2014-12-15 08:00:00.000  345.20  348.67  345.81  345.62   
 3       1418634000  2014-12-15 09:00:00.000  345.81  348.25  347.97  347.39   
 4       1418637600  2014-12-15 10:00:00.000  347.97  347.71  346.98  346.08   
 5       1418641200  2014-12-15 11:00:00.000  346.98  347.58  347.41  346.30   
 6       1418644800  2014-12-15 12:00:00.000  347.41  347.70  346.37  345.35   
 7       1418648400  2014-12-15 13:00:00.000  346.37  347.52  346.78  345.70   
 8       1418652000  2014-12-15 14:00:00.000  346.78  346.14  345.64  344.78   
 9       1418655600  2014-12-15 15:00:00.000  345.64  347.34  346.51  345.21   
 10      1418659200  2014-12-15 16:00:00.000  346.51  347.46  347.39  346.30   
 11      1418662800  2014-12-15 17:00

In [13]:
daily = (
    bitcoin.sort_values("dt")
      .groupby("date")
      .agg(
          open_price=("OPEN", "first"),
          close_price=("CLOSE", "last")
      )
)

daily["next_open"] = daily["open_price"].shift(-1)
daily["next_close"] = daily["close_price"].shift(-1)
daily["diff"] = daily["next_close"] - daily["next_open"]
daily["up_indicator"] = (daily["diff"] > 0).astype(int)



In [14]:
ref_df = daily[["next_open", "next_close", "diff", "up_indicator"]].copy()
ref_df = ref_df.reset_index().rename(columns={"index": "day"})

In [15]:
ref_df.to_csv("reference_data.csv", index=False)

In [17]:
ref_df =  pd.read_csv('reference_data.csv')

In [18]:
ref_df

,date,next_open,next_close,diff,up_indicator
0,2014-12-15,345.21,331.49,-13.72,0
1,2014-12-16,331.49,320.02,-11.47,0
2,2014-12-17,320.02,310.96,-9.06,0
3,2014-12-18,310.96,317.70,6.74,1
4,2014-12-19,317.70,330.39,12.69,1
...,...,...,...,...,...
3992,2025-11-19,91483.02,88067.49,-3415.53,0
3993,2025-11-20,86553.58,84237.34,-2316.24,0
3994,2025-11-21,85087.62,84696.98,-390.64,0
3995,2025-11-22,84696.98,84653.66,-43.32,0


In [ ]:
# now lets gt daily photos 

In [19]:
import plotly.graph_objects as go
from PIL import Image
import matplotlib.pyplot as plt
import plotly.express as px

num_days = len(ref_df)

for i in range(3680,num_days):
  current_df = df_dict[i]
  fig = go.Figure(data=[go.Candlestick(x=current_df['DATETIME'],
                                       open=current_df['OPEN'],
                                       high=current_df['HIGH'],
                                       low=current_df['LOW'],
                                       close=current_df['CLOSE'])])

  fig.update_yaxes(dtick=20, showgrid=True, gridcolor="lightgray")
  fig.write_image(f"fig_{i}.png")